In [ ]:
import geopandas as gpd
import numpy as np

# Load shapefiles
mesh = gpd.read_file("/home/stagiaire/Téléchargements/PR/D/mesh_rj.shp")

# Assign labels based on conditions
mesh['label'] = np.where(
    (mesh['vegetation'] <= 0.95) & 
    (mesh['ghsl'] >= 0.5) & 
    (mesh['osm'] <= 0.5) & 
    (mesh['favelas'] > 0.9),
    1, 
    np.where(
        (mesh['vegetation'] <= 0.95) & 
        (mesh['ghsl'] >= 0.5) & 
        (mesh['osm'] <= 0.5) & 
        (mesh['favelas'] == 0),
        0, 
        np.nan 
    )
)

# Filter dataset to retain only labeled data
dataset = mesh[mesh['label'].notna()].copy()

# Load zones shapefile
zones = gpd.read_file("/home/stagiaire/Téléchargements/PR/D/zones.shp")

# Assign each dataset entry to a zone
dataset['centroid'] = dataset.geometry.centroid
points_zones = gpd.sjoin(dataset.set_geometry('centroid'), zones[['fid', 'geometry']], how='left', predicate='within')
dataset['zone'] = points_zones['fid']
dataset = dataset.drop(columns=['centroid'])
dataset = dataset[dataset['zone'].notna()]

# Print statistics for each zone
for i in dataset['zone'].unique():
    count_1 = len(dataset[(dataset['zone'] == i) & (dataset['label'] == 1)])
    ratio_1 = count_1 / len(dataset[dataset['label'] == 1])
    print(f"Zone {int(i)}: {count_1} {ratio_1:.2f}")
print("\n")

for i in dataset['zone'].unique():
    count_0 = len(dataset[(dataset['zone'] == i) & (dataset['label'] == 0)])
    ratio_0 = count_0 / len(dataset[dataset['label'] == 0])
    print(f"Zone {int(i)}: {count_0} {ratio_0:.2f}")
print("\n")

# Print total counts of each label
print(len(dataset[dataset['label'] == 0]))
print(len(dataset[dataset['label'] == 1]))